# Baseline Retrieval vs. Cross-Encoder Re-Ranking

Two retrieval pipelines over **one** chunked corpus — the *Twilight Imperium 4th Edition
Living Rules Reference* (v1.3, official FFG PDF).

* **Pipeline 1 (baseline)** — bi-encoder embeddings in a FAISS index → **Top 10** by cosine
  similarity → **Top 3** straight into the prompt → **Answer 1**.
* **Pipeline 2 (re-ranking)** — the **same Top 10** → all 10 `(query, chunk)` pairs scored by
  a cross-encoder → re-sorted → **Top 3** → the **identical prompt template** → **Answer 2**.

Everything downstream (rank shifts, answer comparison, latency, scaling) is written to
`results/` as CSV so the report can cite a file rather than a screenshot.
Cell-by-cell reasoning and the gotchas live in [docs/CELL_GUIDE.md](../docs/CELL_GUIDE.md).

## 0. Setup

Configuration, corpus download, the two local models, and the Gemini client.

In [1]:
from pathlib import Path
import hashlib, json, os, platform, re, statistics, time

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA, RESULTS = ROOT / "data", ROOT / "results"
DATA.mkdir(exist_ok=True); RESULTS.mkdir(exist_ok=True)

PDF_URL  = ("https://images-cdn.fantasyflightgames.com/filer_public/da/df/"
            "dadf9f07-78f3-43ac-9cce-dd6b55b24ec2/ti4_living_rules_reference_v1_3_web.pdf")
PDF_PATH = DATA / "ti4_living_rules_reference_v1_3_web.pdf"

# both are 6-layer MiniLM, so comparable size (not a clean architecture-only comparison)
BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"

# model ids expire, so the smoke test below picks the first candidate that answers
GEN_MODEL_CANDIDATES = ["gemini-3.5-flash", "gemini-3.1-flash-lite"]
GEN_MODEL = None

TOP_K_RETRIEVE = 10
TOP_K_CONTEXT  = 3

MAX_CHUNK_TOKENS   = 230
TOKENIZER_HARD_CAP = 256              # all-MiniLM-L6-v2 truncates here
OVERLAP_UNITS      = 1

LLM_CALL_BUDGET = 26              # 12 queries x 2 answers + smoke test, from an empty cache
CACHE_PATH      = RESULTS / "answers_cache.json"

NEAR_DUP_COSINE = 0.90
LATENCY_REPEATS = 10

print(f"root={ROOT}")
print(f"python={platform.python_version()}  machine={platform.machine()}  "
      f"system={platform.system()} {platform.release()}")

root=/Users/d400l/Desktop/LMA_RAG/LMA_RAG
python=3.12.12  machine=arm64  system=Darwin 25.4.0


In [2]:
import urllib.request

# data/*.pdf is gitignored, so fetch the corpus rather than ship it
if not PDF_PATH.exists():
    print("downloading corpus ...")
    urllib.request.urlretrieve(PDF_URL, PDF_PATH)
print(f"{PDF_PATH.name}: {PDF_PATH.stat().st_size:,} bytes")

ti4_living_rules_reference_v1_3_web.pdf: 3,313,611 bytes


In [3]:
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
bi_encoder    = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

HARDWARE = (f"{platform.machine()} / {platform.system()} {platform.release()} / "
            f"torch {torch.__version__} / device={DEVICE}")
print(HARDWARE)
print("bi-encoder max_seq_length:", bi_encoder.max_seq_length,
      "| embedding dim:", bi_encoder.get_sentence_embedding_dimension())

def count_tokens(text: str) -> int:
    # the model's own tokenizer; char or word counts would miss the 256 word-piece cap
    return len(bi_encoder.tokenizer(text)["input_ids"])

/Users/d400l/Desktop/LMA_RAG/LMA_RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


arm64 / Darwin 25.4.0 / torch 2.5.1 / device=mps
bi-encoder max_seq_length: 256 | embedding dim: 384


In [4]:
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv(ROOT / ".env")
assert os.getenv("GEMINI_API_KEY"), "GEMINI_API_KEY missing — copy .env.example to .env"
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])      # the key is never printed

# temperature stays unset: gemini 3 is tuned around its defaults, so the "only context differs"
# claim rests on the fixed prompt template plus the on-disk answer cache instead
THINKING_CFG = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(thinking_level="MINIMAL"))
PLAIN_CFG    = types.GenerateContentConfig()

LLM_CALLS  = 0            # successful generations only
_exhausted = set()        # models whose quota is spent
GEN_CONFIG = None         # frozen after the first success
RPM_PAUSE_S     = 4.0
MAX_RETRY       = 3
MAX_WAIT_S      = 30.0
_last_call_ts   = 0.0
# 429 means either a per-minute limit or a spent quota; only the first is worth retrying
TRANSIENT = ("429", "RESOURCE_EXHAUSTED", "500", "502", "503", "504",
             "UNAVAILABLE", "overloaded", "high demand", "INTERNAL")

def _retry_delay(msg: str, attempt: int) -> float:
    m = re.search(r'retryDelay[\"\':\s]+(\d+(?:\.\d+)?)s', msg)
    return min(float(m.group(1)) + 1 if m else 15.0 * (attempt + 1), MAX_WAIT_S)

def _is_quota_exhaustion(msg: str) -> bool:
    return "exceeded your current quota" in msg or "quota_metric" in msg

def _raw_call(model: str, prompt: str):
    global LLM_CALLS, GEN_CONFIG, _last_call_ts
    if model in _exhausted:
        raise RuntimeError(f"{model}: quota already known exhausted this run")
    variants = [GEN_CONFIG] if GEN_CONFIG else [("minimal-thinking", THINKING_CFG),
                                                ("default-thinking", PLAIN_CFG)]
    errors = []
    for name, cfg in variants:
        for attempt in range(MAX_RETRY + 1):
            if LLM_CALLS >= LLM_CALL_BUDGET:
                raise RuntimeError(f"LLM call budget of {LLM_CALL_BUDGET} exhausted")
            gap = RPM_PAUSE_S - (time.time() - _last_call_ts)
            if gap > 0:
                time.sleep(gap)
            t0 = time.perf_counter()
            try:
                r = client.models.generate_content(model=model, contents=prompt, config=cfg)
                dt = time.perf_counter() - t0      # request only, so backoff can't skew latency
                _last_call_ts = time.time()
                LLM_CALLS += 1
                GEN_CONFIG = (name, cfg)
                return (r.text or "").strip(), dt
            except Exception as e:
                _last_call_ts = time.time()
                msg = str(e)
                if _is_quota_exhaustion(msg):
                    _exhausted.add(model)          # no point confirming it a second time
                    raise RuntimeError(f"quota exhausted for {model}") from None
                if any(t in msg for t in TRANSIENT):
                    wait = _retry_delay(msg, attempt)
                    print(f"    transient failure on {model} ({type(e).__name__}); waiting "
                          f"{wait:.0f}s (attempt {attempt + 1}/{MAX_RETRY + 1})")
                    time.sleep(wait)
                    continue
                errors.append(f"{name}: {type(e).__name__}: {msg[:160]}")
                break                     # a real error, not transient: try the next variant
        else:
            _exhausted.add(model)      # stop hammering it for the rest of the run
            errors.append(f"{name}: still failing after {MAX_RETRY + 1} attempts")
    raise RuntimeError("generation failed:\n  " + "\n  ".join(errors))

# smoke test: resolve GEN_MODEL to the first candidate that actually answers
for candidate in GEN_MODEL_CANDIDATES:
    try:
        reply, _ = _raw_call(candidate, "Reply with the single word: ready")
        GEN_MODEL = candidate
        print(f"GEN_MODEL={GEN_MODEL!r} [{GEN_CONFIG[0]}] replied {reply[:40]!r}")
        break
    except Exception as e:
        print(f"{candidate!r} unavailable: {str(e)[:220]}")
# not fatal: every answer is in the committed cache, so a run that needs no new generation must
# still work when the key is out of quota or the model is briefly down. generate() raises only
# if an uncached answer is actually required.
if GEN_MODEL is None:
    GEN_MODEL = GEN_MODEL_CANDIDATES[0]
    print(f"WARNING: no candidate responded; continuing with cached answers only "
          f"(GEN_MODEL={GEN_MODEL!r} is a placeholder for cache lookups)")
print(f"temperature sent: {THINKING_CFG.temperature}  (None = provider default)")
print(f"LLM calls used by setup: {LLM_CALLS}/{LLM_CALL_BUDGET}")


def advance_model() -> bool:
    global GEN_MODEL
    rest = GEN_MODEL_CANDIDATES[GEN_MODEL_CANDIDATES.index(GEN_MODEL) + 1:]
    for cand in rest:
        try:
            _raw_call(cand, "Reply with the single word: ready")
            print(f"  failed over: GEN_MODEL {GEN_MODEL!r} -> {cand!r}")
            GEN_MODEL = cand
            return True
        except Exception as e:
            print(f"  candidate {cand!r} also unavailable: {str(e)[:120]}")
    return False

'gemini-3.5-flash' unavailable: quota exhausted for gemini-3.5-flash


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 15s (attempt 1/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 2/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 3/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 4/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 15s (attempt 1/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 2/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 3/4)


    transient failure on gemini-3.1-flash-lite (ServerError); waiting 30s (attempt 4/4)


'gemini-3.1-flash-lite' unavailable: generation failed:
  minimal-thinking: still failing after 4 attempts
  default-thinking: still failing after 4 attempts
temperature sent: None  (None = provider default)
LLM calls used by setup: 0/26


## 1. Corpus and chunking

Chunks follow the rulebook's own numbered rule boundaries, so every one is citable as
"Rule 58.6, p.21". Only pages 6-31 (the numbered rules), card art filtered out by block
geometry, `RELATED TOPICS:` keyword lists dropped — reasoning in
[docs/CELL_GUIDE.md](../docs/CELL_GUIDE.md).

In [5]:
import fitz   # PyMuPDF

RULES_PAGES  = range(5, 31)                        # 0-indexed; printed pages 6-31
BODY_MARGINS = (36.0, 54.0, 315.0, 333.0)          # left body, left bullet, right body, right bullet
COL_SPLIT    = 306.0

_LEAD = re.compile(r'^(\d{1,2}(?:\.\d{1,2}[a-z]?)?)\t\s*(.*)$', re.S)

def _norm(t: str) -> str:
    t = t.replace(' ', ' ').replace('\xa0', ' ').replace(' ', ' ')
    t = t.replace('\t', '\x01')                    # rule number is tab-separated, so protect it
    t = re.sub(r'\s*\n\s*', ' ', t)                # line breaks are layout, not meaning
    t = re.sub(r'[ ]+', ' ', t).strip()
    return t.replace('\x01', '\t')

def body_blocks(page):
    out = []
    for x0, y0, x1, y1, txt, *_ in page.get_text("blocks"):
        if y0 > 740 or y1 < 45 or not txt.strip():        # footer / header
            continue
        in_col      = (30 <= x0 <= 300) or (310 <= x0 <= 580)
        near_margin = any(abs(x0 - m) <= 4 for m in BODY_MARGINS)
        if in_col and (near_margin or (x1 - x0) >= 120):  # narrow stray blocks are card art
            out.append((0 if x0 < COL_SPLIT else 1, y0, _norm(txt)))
    out.sort(key=lambda r: (r[0], r[1]))               # pymupdf's order is not reading order
    return [t for _, _, t in out]

def parse_units(doc):
    units, stats = [], {"related_topics_dropped": 0, "orphan_blocks": 0}
    sec = title = cur = None

    def flush():
        nonlocal cur
        if cur and cur["text"].strip():
            units.append(cur)
        cur = None

    for pno in RULES_PAGES:
        for txt in body_blocks(doc[pno]):
            if txt.startswith("RELATED TOPICS"):
                stats["related_topics_dropped"] += 1
                continue
            m = _LEAD.match(txt)
            if m and "." not in m.group(1):                      # "9  ANOMALIES"
                flush()
                sec, title = m.group(1), (m.group(2).strip() or f"SECTION {m.group(1)}")
            elif m:                                              # "9.1  An anomaly is ..."
                flush()
                cur = {"sec": sec, "title": title, "rule_no": m.group(1),
                       "text": m.group(2).replace("\t", " ").strip(), "page": pno + 1}
            elif cur is not None:                                # bullet or continuation
                cur["text"] += " " + txt.replace("\t", " ")
            elif sec is not None:                                # section preamble -> "<sec>.0"
                cur = {"sec": sec, "title": title, "rule_no": f"{sec}.0",
                       "text": txt.replace("\t", " "), "page": pno + 1}
            else:
                stats["orphan_blocks"] += 1                      # front matter before Rule 1
    flush()
    return units, stats

doc = fitz.open(PDF_PATH)
units, parse_stats = parse_units(doc)
print(f"pages={doc.page_count}  rule units={len(units)}  "
      f"sections={len({(u['sec'], u['title']) for u in units})}  {parse_stats}")

pages=38  rule units=532  sections=89  {'related_topics_dropped': 89, 'orphan_blocks': 2}


In [6]:
def _rule_key(r):
    a, _, b = r.partition(".")
    return (int(a), int(re.sub(r'\D', '', b) or 0), b)

def build_chunks(units, max_tokens=MAX_CHUNK_TOKENS, overlap=OVERLAP_UNITS):
    # the section title is prefixed because a sub-rule is often one clause whose subject sits
    # in the heading; overlap=1 keeps a rule with its neighbour, at the cost of near-duplicates
    by_sec = {}
    for u in units:
        by_sec.setdefault((u["sec"], u["title"]), []).append(u)

    def render(sec, title, group):
        body = "\n".join(f'{u["rule_no"]} {u["text"]}' for u in group)
        return f"{title} (Rule {sec})\n{body}"

    def split_oversize(sec, title, u):
        # keeps the rule number, so gold-chunk matching still works
        if count_tokens(render(sec, title, [u])) <= max_tokens:
            return [u]
        parts, buf = [], ""
        for sent in re.split(r'(?<=[.!?]) +', u["text"]):
            trial = (buf + " " + sent).strip()
            if buf and count_tokens(render(sec, title, [dict(u, text=trial)])) > max_tokens:
                parts.append(dict(u, text=buf)); buf = sent
            else:
                buf = trial
        if buf:
            parts.append(dict(u, text=buf))
        return parts

    packed = []
    for (sec, title), us in by_sec.items():
        us = [p for u in us for p in split_oversize(sec, title, u)]
        i, group = 0, []
        while i < len(us):
            if group and count_tokens(render(sec, title, group + [us[i]])) > max_tokens:
                packed.append((sec, title, list(group)))
                tail = group[-overlap:] if overlap else []
                if tail and count_tokens(render(sec, title, tail + [us[i]])) > max_tokens:
                    tail = []                                    # overlap itself would overflow
                group = tail
                continue
            group = group + [us[i]]
            i += 1
        if group:
            packed.append((sec, title, list(group)))

    rows = []
    for sec, title, group in packed:
        text = render(sec, title, group)
        rows.append({"section_no": int(sec), "section_title": title,
                     "rule_from": group[0]["rule_no"], "rule_to": group[-1]["rule_no"],
                     "rule_nos": ";".join(u["rule_no"] for u in group),
                     "page_from": min(u["page"] for u in group),
                     "page_to": max(u["page"] for u in group),
                     "n_units": len(group), "n_tokens": count_tokens(text), "text": text})
    rows.sort(key=lambda r: (r["section_no"], _rule_key(r["rule_from"])))
    for cid, r in enumerate(rows):
        r["chunk_id"] = cid
    return rows

import pandas as pd

chunk_rows = build_chunks(units)
chunks = pd.DataFrame(chunk_rows)[["chunk_id", "section_no", "section_title", "rule_from",
                                   "rule_to", "rule_nos", "page_from", "page_to", "n_units",
                                   "n_tokens", "text"]]

# nothing may reach the 256 word-piece cap, and .loc[chunk_id] needs index == chunk_id
assert chunks["n_tokens"].max() < TOKENIZER_HARD_CAP, "a chunk would be silently truncated"
assert (chunks.index == chunks["chunk_id"]).all(), "chunk_id must equal the row index"
chunks.to_csv(RESULTS / "chunks.csv", index=False)

print(f"chunks={len(chunks)}  tokens: min={chunks.n_tokens.min()} "
      f"median={int(chunks.n_tokens.median())} max={chunks.n_tokens.max()} "
      f"(hard cap {TOKENIZER_HARD_CAP})")
print(chunks.head(3)[["chunk_id", "section_title", "rule_from", "rule_to", "page_from", "n_tokens"]]
      .to_string(index=False))
print("\nexample chunk:\n" + chunks.loc[0, "text"][:400])

Token indices sequence length is longer than the specified maximum sequence length for this model (267 > 256). Running this sequence through the model will result in indexing errors


chunks=172  tokens: min=20 median=179 max=229 (hard cap 256)
 chunk_id section_title rule_from rule_to  page_from  n_tokens
        0     ABILITIES       1.0     1.6          6       211
        1     ABILITIES       1.6    1.11          6       185
        2     ABILITIES      1.11    1.15          6       200

example chunk:
ABILITIES (Rule 1)
1.0 Cards and faction sheets each have abilities that players can resolve to trigger various game effects.
1.1 Each ability describes when and how a player can resolve it. •  If an ability with a specified duration is resolved, the effect of the ability remains through that duration, even if the component that caused the ability is removed.
1.2 If a card has multiple abilities, 


## 2. Pipeline 1 — baseline retrieval (bi-encoder)

Chunks are embedded independently and stored in a FAISS `IndexFlatIP`. Top 10 by cosine, then
Top 3 straight into the prompt as Answer 1.

In [7]:
import faiss, numpy as np

corpus_texts = chunks["text"].tolist()
corpus_emb = bi_encoder.encode(corpus_texts, batch_size=64, convert_to_numpy=True,
                               normalize_embeddings=True, show_progress_bar=False)
index = faiss.IndexFlatIP(corpus_emb.shape[1])     # normalised, so inner product == cosine
index.add(corpus_emb.astype(np.float32))
print(f"indexed {index.ntotal} chunks, dim={corpus_emb.shape[1]}, metric=inner product "
      f"on L2-normalised vectors (= cosine)")

def baseline_retrieve(query: str, k: int = TOP_K_RETRIEVE):
    # this candidate list is the one pipeline 2 re-ranks
    q = bi_encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, ids = index.search(q.astype(np.float32), k)
    return [{"chunk_id": int(c), "bi_score": float(s), "bi_rank": r + 1}
            for r, (c, s) in enumerate(zip(ids[0], scores[0]))]

indexed 172 chunks, dim=384, metric=inner product on L2-normalised vectors (= cosine)


In [8]:
# one template for both pipelines; only the three excerpts differ. "only from the excerpts"
# is what makes a missing chunk show up as an omission instead of being filled in from priors
PROMPT_TEMPLATE = """You are answering a rules question about the board game Twilight Imperium \
(4th Edition). Use ONLY the numbered rule excerpts below. Do not use outside knowledge. If the \
excerpts do not contain the answer, reply exactly: "The provided rules do not cover this."

RULE EXCERPTS
{context}

QUESTION: {question}

Answer in at most four sentences, and cite the rule numbers you relied on."""

def build_prompt(question: str, chunk_ids) -> str:
    ctx = "\n\n".join(f"[excerpt {i}]\n{corpus_texts[cid]}" for i, cid in enumerate(chunk_ids, 1))
    return PROMPT_TEMPLATE.format(context=ctx, question=question)

# keyed by (model, prompt hash), so a re-run costs no quota and identical prompts collapse
_cache = json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}

def _cache_key(model: str, prompt: str) -> str:
    return hashlib.sha256(f"{model}\x00{prompt}".encode()).hexdigest()

def _cache_lookup(prompt: str, strict: bool = False):
    # current model first, then the others: answers made before a quota fail-over stay usable
    order = [GEN_MODEL] if strict else \
            [GEN_MODEL] + [m for m in GEN_MODEL_CANDIDATES if m != GEN_MODEL]
    for m in order:
        k = _cache_key(m, prompt)
        if k in _cache:
            return k, _cache[k]
    return None, None

def generate(question: str, chunk_ids, strict_model: bool = False):
    # if re-ranking leaves the top 3 unchanged, the prompt is identical and answer 2 is answer 1
    prompt = build_prompt(question, chunk_ids)
    hit_key, hit = _cache_lookup(prompt, strict=strict_model)
    if hit:
        return hit["answer"], prompt, True, 0.0, hit["model"]
    key = _cache_key(GEN_MODEL, prompt)
    try:
        answer, dt = _raw_call(GEN_MODEL, prompt)   # dt excludes any waiting
    except RuntimeError:
        if not advance_model():                     # quota gone: use whichever model responds
            raise
        key = _cache_key(GEN_MODEL, prompt)
        answer, dt = _raw_call(GEN_MODEL, prompt)
    _cache[key] = {"model": GEN_MODEL, "cache_key": key, "answer": answer,
                   "n_context_chunks": len(chunk_ids), "seconds": round(dt, 3)}
    CACHE_PATH.write_text(json.dumps(_cache, indent=2))
    return answer, prompt, False, dt, GEN_MODEL

## 3. Pipeline 2 — cross-encoder re-ranking

The same 10 candidates, each `(query, chunk)` pair scored jointly by the cross-encoder, then
re-sorted. Scores are raw logits, comparable within one query's list but not across queries.

In [9]:
def rerank(query: str, candidates):
    # same 10 candidates as the baseline; this never retrieves anything itself
    pairs = [(query, corpus_texts[c["chunk_id"]]) for c in candidates]
    ce_scores = cross_encoder.predict(pairs, show_progress_bar=False)
    out = [dict(c, ce_score=float(s)) for c, s in zip(candidates, ce_scores)]
    out.sort(key=lambda c: -c["ce_score"])
    for r, c in enumerate(out, 1):
        c["ce_rank"] = r
    return out

## 4. Query set

Ten questions, each with its gold rule numbers written down before any retrieval ran, and
phrased the way a player would ask rather than quoting the rulebook's own headings. Q9 and Q10
came out of the hunt in the next section.

In [10]:
QUERIES = [
    {"qid": "Q1", "category": "keyword_trap",
     "question": "How many cards is a player allowed to keep in their hand between rounds?",
     "gold_rules": ["2.4"],
     "why": "'cards' and 'hand' collide with promissory-note and objective-card passages; the "
            "hand limit itself lives in ACTION CARDS and never uses the word 'round'."},

    {"qid": "Q2", "category": "synonym_gap",
     "question": "Can a warship fly into a star that has exploded?",
     "gold_rules": ["75.0", "75.1"],
     "why": "The rule says 'supernova' and 'ship'; the query says 'star that has exploded' and "
            "'warship'. Zero lexical overlap with the gold passage."},

    {"qid": "Q3", "category": "exception_corner",
     "question": "May a player build new warships at a shipyard while an enemy fleet sits in "
                 "the same system?",
     "gold_rules": ["58.6", "14.0", "14.1"],
     "why": "The restriction is stated twice in different vocabulary — PRODUCING UNITS 58.6 and "
            "BLOCKADED 14.1 — and both carry the 'ground forces can still be produced' exception."},

    {"qid": "Q4", "category": "negation_constraint",
     "question": "Can I take a tactical action on a system I have already claimed this round?",
     "gold_rules": ["5.2"],
     "why": "A prohibition ('cannot activate a system that already contains one of their "
            "command tokens') phrased by the player as a permission question."},

    {"qid": "Q5", "category": "role_limit",
     "question": "How many planetary defence systems can one player have on the same world?",
     "gold_rules": ["54.3", "74.5"],
     "why": "A numeric cap duplicated across PDS and STRUCTURES; 'world' vs 'planet' and "
            "'planetary defence system' vs 'PDS' are both paraphrases."},

    {"qid": "Q6", "category": "exception_corner",
     "question": "Who may remove the token blocking the galaxy's central planet, and what does "
                 "it cost?",
     "gold_rules": ["46.1", "26.2"],
     "why": "'central planet' means Mecatol Rex and the token is the custodians token; the cost "
            "(six influence) sits inside an 'unless' clause. Gold originally listed only 46.1 "
            "(MECATOL REX), which merely mentions the token - that scored as 'gold never "
            "retrieved' while both answers were correct. 26.2 (CUSTODIANS TOKEN) is the "
            "authoritative statement of the cost, so both are gold."},

    {"qid": "Q7", "category": "negation_constraint",
     "question": "When is a player forbidden from passing their turn?",
     "gold_rules": ["3.2", "3.4"],
     "why": "'forbidden' vs 'cannot pass until'; the player-count exception for three- and "
            "four-player games is a bullet under the general rule."},

    {"qid": "Q8", "category": "exception_corner",
     "question": "If a unit takes damage, does it fight less effectively?",
     "gold_rules": ["76.2"],
     "why": "The answer is a negation plus an exception ('functionally the same, except that it "
            "cannot use Sustain Damage') inside a section about marking damage. Relabelled from "
            "buried_vital_chunk: its gold sat at #2 before and after, so it demonstrates an "
            "exception clause, not a buried chunk. The buried-chunk phenomenon is a rank "
            "property, shown by the promotions Q4/Q9/Q10, not by any one question type."},

    # Q9 and Q10 were not guessed: they come from the retrieval-only hunt in §4b below, which
    # searched 16 candidate questions for gold chunks that the bi-encoder buries at ranks 4-10.
    {"qid": "Q9", "category": "role_limit",
     "question": "How many swaps am I allowed with the player beside me during one turn?",
     "gold_rules": ["83.1"],
     "why": "The cap lives in TRANSACTIONS 83.1 ('up to one transaction with each of their "
            "neighbors'), but 'swaps' and 'player beside me' pull the bi-encoder towards DEALS "
            "and NEIGHBORS, which define the relationship without stating the limit."},

    {"qid": "Q10", "category": "negation_constraint",
     "question": "What stops my factory from turning out new warships?",
     "gold_rules": ["14.0", "14.1"],
     "why": "'Factory' is not a term the rulebook uses, and the answer is a prohibition in "
            "BLOCKADED 14.1. Contrast with Q3, which asks the same thing in a compound way and "
            "gets no promotion — query specificity, not just vocabulary, moves the ranking."},

    # Q11 and Q12 close the two category gaps found by auditing the earlier ten queries.
    {"qid": "Q11", "category": "keyword_trap",
     "question": "Do I still get the secondary ability of other players' strategy cards after I "
                 "have passed?",
     "gold_rules": ["3.3"],
     "why": "The clearest keyword trap in the document. 'secondary ability' and 'strategy "
            "cards' are the vocabulary of STRATEGY CARD and of all eight strategy-card "
            "sections, which is where the bi-encoder goes; the answer is a bullet under ACTION "
            "PHASE 3.3 about passing, which shares almost no wording with the question."},

    {"qid": "Q12", "category": "exception_corner",
     "question": "I won the space battle but brought no troops along. Can I capture the planet?",
     "gold_rules": ["44.2"],
     "why": "An 'except when' in practice: winning space combat is not enough, INVASION 44.2 "
            "requires ground forces to be committed. Found by the hunt below as a case where "
            "the gold sits at rank 4 and the cross-encoder promotes it."},
]
print(f"{len(QUERIES)} queries; LLM budget {LLM_CALL_BUDGET}, used so far {LLM_CALLS}")

def gold_chunk_ids(gold_rules):
    # a set, not one chunk: overlap and the rulebook's own repetition put a rule in two places
    want = set(gold_rules)
    return {int(r.chunk_id) for r in chunks.itertuples()
            if want & set(str(r.rule_nos).split(";"))}

for q in QUERIES:
    q["gold_chunk_ids"] = gold_chunk_ids(q["gold_rules"])
    assert q["gold_chunk_ids"], f'{q["qid"]}: no chunk contains rules {q["gold_rules"]}'
    print(f'{q["qid"]} gold rules {q["gold_rules"]} -> chunks {sorted(q["gold_chunk_ids"])}')

12 queries; LLM budget 26, used so far 0
Q1 gold rules ['2.4'] -> chunks [6]
Q2 gold rules ['75.0', '75.1'] -> chunks [140]
Q3 gold rules ['58.6', '14.0', '14.1'] -> chunks [27, 105]
Q4 gold rules ['5.2'] -> chunks [11]
Q5 gold rules ['54.3', '74.5'] -> chunks [95, 139]
Q6 gold rules ['46.1', '26.2'] -> chunks [49, 79]
Q7 gold rules ['3.2', '3.4'] -> chunks [8, 9]
Q8 gold rules ['76.2'] -> chunks [141]
Q9 gold rules ['83.1'] -> chunks [160]
Q10 gold rules ['14.0', '14.1'] -> chunks [27]
Q11 gold rules ['3.3'] -> chunks [8, 9]
Q12 gold rules ['44.2'] -> chunks [74, 75]


## 4b. How the failure queries were found (retrieval only, no LLM calls)

The first eight queries produced only one clean promotion, so 16 further candidates were
screened through both pipelines, with gold identified by the authoritative sentence rather than
a remembered rule number. This costs no quota. Q9 and Q10 came from here; the no-ops and the
one rank regression stay in the table.

In [11]:
HUNT_CANDIDATES = [
    ("H1",  "transport",           "If a ship carrying troops is destroyed, what happens to the troops?",
     r"fighters and ground forces being transported.{0,80}removed from the board"),
    ("H2",  "keyword_trap",        "Can I strike a bargain with a player sitting on the far side of the galaxy?",
     r"deals with each other at any time, even if they are not neighbors"),
    ("H3",  "role_limit",          "How many swaps am I allowed with the player beside me during one turn?",
     r"up to one transaction with each of their neighbors"),
    ("H4",  "role_limit",          "Is there a cap on how many victory conditions I can claim in a single round?",
     r"maximum of one public objective and one secret objective"),
    ("H5",  "negation_constraint", "Can my foot soldiers walk from one world to another by themselves?",
     r"Ground forces.{0,120}capacity values, or being transported"),
    ("H6",  "exception_corner",    "What do I have to roll to keep my vessel from being sucked into the rift?",
     r"gravity rift.{0,200}destroyed"),
    ("H7",  "exception_corner",    "At what point is someone knocked out of the game for good?",
     r"A player is eliminated when they meet all"),
    ("H8",  "exception_corner",    "Can I pull my fleet out of a battle once shooting has started?",
     r"ANNOUNCE RETREATS"),
    ("H9",  "keyword_trap",        "What limits the number of warships I can keep in one system?",
     r"fleet pool"),
    ("H10", "negation_constraint", "What stops my factory from turning out new warships?",
     r"blockaded.{0,120}produce ships"),
    ("H11", "exception_corner",    "If two abilities go off at the same instant, which one resolves first?",
     r"abilit(?:y|ies).{0,120}(?:same time|simultaneous)"),
    ("H12", "negation_constraint", "May I hand over a promissory note I already gave away once?",
     r"maximum of one promissory note from their hand"),
    ("H13", "synonym_gap",         "Do wrecked vessels go back into my reserves?",
     r"removed from the game board and returned to their reinforcements"),
    ("H14", "keyword_trap",        "How do I take over a world that already belongs to a rival?",
     r"INVASION|commit ground forces"),
    ("H15", "exception_corner",    "Can the central world be invaded before anyone pays the toll?",
     r"custodians token"),
    ("H16", "negation_constraint", "Am I allowed to activate a system twice in the same turn?",
     r"cannot activate a system that already contains"),

    # a second sweep aimed at keyword traps: each question is phrased in the vocabulary of a
    # section that does NOT hold the answer. Q11 and Q12 came from here
    ("K1",  "keyword_trap", "Do I still get the secondary ability of other players' strategy cards after I have passed?",
     r"passed can still resolve the secondary ability"),
    ("K2",  "keyword_trap", "What happens to my strategy card once I have used its main ability?",
     r"exhaust their chosen strategy card"),
    ("K3",  "exception_corner", "I won the space battle but brought no troops along. Can I capture the planet?",
     r"STEP 2—COMMIT GROUND FORCES"),
    ("K4",  "keyword_trap", "Is a planet still mine if I have no units sitting on it?",
     r"can control a planet that they do not have any units on"),
    ("K5",  "keyword_trap", "Does a wormhole make two systems count as neighbours?",
     r"system that has a wormhole is treated as being adjacent"),
    ("K6",  "keyword_trap", "How far is a ship allowed to travel in a single move?",
     r"move value indicates the distance"),
    ("K8",  "negation_constraint", "Can I play two copies of the same action card at the same moment?",
     r"same name cannot be played"),
    ("K13", "keyword_trap", "Can my planetary defence guns fire through a wormhole?",
     r"use the .Space Cannon. abilities of their PDS units through wormholes"),
    ("K19", "negation_constraint", "Can I move ships out of a system that already has my own command token in it?",
     r"cannot move if it started its movement in another system that contains"),
    ("K22", "keyword_trap", "Which step of the round do I draw a new action card in?",
     r"STEP 3—DRAW ACTION CARDS|Each player draws one action card during each status phase"),
    ("K23", "keyword_trap", "How do I get more command tokens during the game?",
     r"spend influence to gain"),
    ("K24", "keyword_trap", "Do fighters count against the number of ships I can keep in a system?",
     r"do not count against that player.s fleet pool"),
    ("K26", "keyword_trap", "What protects a planet from being shelled from space?",
     r"planetary shield"),
    ("L2",  "keyword_trap", "Can an enemy gun platform shoot my fleet while it is flying past?",
     r"SPACE CANNON OFFENSE|Space Cannon Offense"),
    ("L3",  "exception_corner", "What do I do if I want to build a unit but have none left in reserve?",
     r"cannot produce a unit because it is not in their reinforcements"),
    ("L4",  "keyword_trap", "When exactly do I claim victory points for finishing an objective?",
     r"STEP 1—SCORE OBJECTIVES|Score Objectives"),
    ("L5",  "keyword_trap", "Does my shipyard let me place new units on any planet in the system?",
     r"follows the rules of the .Production. ability to determine where"),
    ("L9",  "keyword_trap", "Does my carrier lose its cargo if it gets blown up in a fight?",
     r"removed from the game board and returned to their reinforcements"),
    ("L10", "exception_corner", "Can I keep attacking after my opponent runs out of ships?",
     r"combat ends|continues until"),
    ("L13", "role_limit", "Is there a cap on how many units I can have in total?",
     r"Units are limited to those included in the game"),
]

hunt = []
for hid, cat, question, gold_pat in HUNT_CANDIDATES:
    gold = {int(r.chunk_id) for r in chunks.itertuples()
            if re.search(gold_pat, str(r.text), re.I)}
    cands = baseline_retrieve(question)
    reranked = rerank(question, cands)
    bi_r = min([c["bi_rank"] for c in cands    if c["chunk_id"] in gold], default=None)
    ce_r = min([c["ce_rank"] for c in reranked if c["chunk_id"] in gold], default=None)

    if not gold:
        verdict = "no gold match (regex too strict)"
    elif bi_r is None:
        verdict = "gold never entered Top 10"          # re-ranking cannot repair this
    elif bi_r > TOP_K_CONTEXT and ce_r <= TOP_K_CONTEXT:
        verdict = "promotion"
    elif bi_r <= TOP_K_CONTEXT and ce_r > TOP_K_CONTEXT:
        verdict = "regression"
    else:
        verdict = "no change in Top 3 membership"
    hunt.append({"hid": hid, "category": cat, "question": question,
                 "n_gold_chunks": len(gold), "gold_chunks": ";".join(map(str, sorted(gold))),
                 "gold_rank_baseline": bi_r, "gold_rank_rerank": ce_r, "verdict": verdict,
                 "promoted_to_queryset": hid in ("H3", "H10")})

hunt_df = pd.DataFrame(hunt)
hunt_df.to_csv(RESULTS / "failure_hunt.csv", index=False)
print(hunt_df[["hid", "category", "gold_rank_baseline", "gold_rank_rerank", "verdict",
               "promoted_to_queryset"]].to_string(index=False))
print("\n" + hunt_df.verdict.value_counts().to_string())
print(f"\n{len(hunt_df)} candidates screened at zero API cost; "
      f"{(hunt_df.verdict == 'promotion').sum()} promotions, "
      f"{(hunt_df.verdict == 'regression').sum()} regressions.")

# the §3 question: does each named category actually produce a repairable failure?
print("\npromotions by category (hunt only):")
print(hunt_df.assign(promoted=hunt_df.verdict.eq("promotion"))
             .groupby("category").promoted.agg(["sum", "count"])
             .rename(columns={"sum": "promotions", "count": "candidates"}).to_string())
kw = hunt_df[hunt_df.category == "keyword_trap"]
print(f"\nkeyword traps screened: {len(kw)}; promotions among them: "
      f"{int(kw.verdict.eq('promotion').sum())}")
print(kw[["hid", "gold_rank_baseline", "gold_rank_rerank", "verdict"]].to_string(index=False))

hid            category  gold_rank_baseline  gold_rank_rerank                          verdict  promoted_to_queryset
 H1           transport                 NaN               NaN no gold match (regex too strict)                 False
 H2        keyword_trap                 3.0               3.0    no change in Top 3 membership                 False
 H3          role_limit                 8.0               1.0                        promotion                  True
 H4          role_limit                 5.0              10.0    no change in Top 3 membership                 False
 H5 negation_constraint                 4.0               3.0                        promotion                 False
 H6    exception_corner                 NaN               NaN no gold match (regex too strict)                 False
 H7    exception_corner                 6.0               2.0                        promotion                 False
 H8    exception_corner                 1.0               1.0   

## 5. Running both pipelines

One baseline retrieval per query, handed unchanged to the cross-encoder, both Top 3 sets
through the same prompt template. Every candidate is recorded, so the tables below are derived
rather than retyped.

In [12]:
PENDING_ANSWER = "(not generated: no model available and this prompt is not cached)"

runs, summary = [], []

for q in QUERIES:
    candidates = baseline_retrieve(q["question"])               # Pipeline 1
    reranked   = rerank(q["question"], candidates)              # Pipeline 2, same candidates
    assert {c["chunk_id"] for c in candidates} == {c["chunk_id"] for c in reranked}, \
        "pipeline 2 must re-rank the baseline candidates, not re-retrieve"

    base_top3 = [c["chunk_id"] for c in candidates[:TOP_K_CONTEXT]]
    rr_top3   = [c["chunk_id"] for c in reranked[:TOP_K_CONTEXT]]
    gold      = q["gold_chunk_ids"]

    bi_gold = [c["bi_rank"] for c in candidates if c["chunk_id"] in gold]
    ce_gold = [c["ce_rank"] for c in reranked   if c["chunk_id"] in gold]
    best_bi = min(bi_gold) if bi_gold else None                 # None = gold not in Top 10 at all
    best_ce = min(ce_gold) if ce_gold else None

    in3_bi = best_bi is not None and best_bi <= TOP_K_CONTEXT
    in3_ce = best_ce is not None and best_ce <= TOP_K_CONTEXT
    if   in3_ce and not in3_bi: outcome = "promotion"
    elif in3_bi and not in3_ce: outcome = "regression"
    elif base_top3 == rr_top3:  outcome = "no_op_identical"
    elif set(base_top3) == set(rr_top3): outcome = "no_op_reordered"
    else: outcome = "changed_context_same_verdict"

    try:
        ans1, p1, hit1, t1, m1 = generate(q["question"], base_top3)     # Answer 1
        ans2, p2, hit2, t2, m2 = generate(q["question"], rr_top3)       # Answer 2
    except RuntimeError as e:
        # no model available and this pair is not cached yet: keep the retrieval evidence for
        # this query rather than losing the whole run, and mark the answers as pending
        print(f'    {q["qid"]}: answers not generated ({str(e)[:70]})')
        ans1 = ans2 = PENDING_ANSWER
        p1, p2 = build_prompt(q["question"], base_top3), build_prompt(q["question"], rr_top3)
        hit1 = hit2 = False
        t1 = t2 = 0.0
        m1 = m2 = "unavailable"
    # a model can run out of quota between the two calls; comparing answers from two models
    # would measure the model, not the context, so answer 1 is redone on the surviving one
    if m1 != m2:
        # align the pair on one model: prefer a model that already has BOTH answers cached
        # (a stale entry from an aborted run can otherwise win the lookup for one half only)
        for m in (m2, m1):
            k1, k2 = _cache_key(m, p1), _cache_key(m, p2)
            if k1 in _cache and k2 in _cache:
                ans1, m1, hit1, t1 = _cache[k1]["answer"], m, True, 0.0
                ans2, m2, hit2, t2 = _cache[k2]["answer"], m, True, 0.0
                break
        else:
            print(f'    {q["qid"]}: model failed over mid-query ({m1} -> {m2}); '
                  f"regenerating Answer 1 on {m2}")
            ans1, p1, hit1, t1, m1 = generate(q["question"], base_top3, strict_model=True)
    assert m1 == m2, f'{q["qid"]}: Answer 1 from {m1}, Answer 2 from {m2}'
    # mask each prompt's own excerpts: both must reduce to the same skeleton
    def _skeleton(prompt, cids):
        for cid in cids:
            prompt = prompt.replace(corpus_texts[cid], "<EXCERPT>")
        return prompt
    assert _skeleton(p1, base_top3) == _skeleton(p2, rr_top3), "prompt template diverged"

    for c in reranked:
        runs.append({"qid": q["qid"], "category": q["category"], "chunk_id": c["chunk_id"],
                     "rule_nos": chunks.loc[c["chunk_id"], "rule_nos"],
                     "section_title": chunks.loc[c["chunk_id"], "section_title"],
                     "page_from": chunks.loc[c["chunk_id"], "page_from"],
                     "bi_rank": c["bi_rank"], "bi_score": round(c["bi_score"], 4),
                     "ce_rank": c["ce_rank"], "ce_score": round(c["ce_score"], 4),
                     "is_gold": c["chunk_id"] in gold,
                     "in_baseline_top3": c["chunk_id"] in base_top3,
                     "in_rerank_top3": c["chunk_id"] in rr_top3})

    summary.append({"qid": q["qid"], "category": q["category"], "question": q["question"],
                    "gold_rules": ";".join(q["gold_rules"]),
                    "gold_rank_baseline": best_bi, "gold_rank_rerank": best_ce,
                    "gold_in_top3_baseline": in3_bi, "gold_in_top3_rerank": in3_ce,
                    "outcome": outcome,
                    "baseline_top3": ";".join(map(str, base_top3)),
                    "rerank_top3": ";".join(map(str, rr_top3)),
                    "answer1": ans1, "answer2": ans2, "answer_model": m1,
                    "answers_identical": ans1 == ans2 and m1 != "unavailable",
                    "answers_generated": m1 != "unavailable",
                    "a1_cache_hit": hit1, "a2_cache_hit": hit2,
                    "a1_seconds": t1, "a2_seconds": t2})
    print(f'{q["qid"]} {q["category"]:<24} gold rank {str(best_bi):>4} -> {str(best_ce):>4}  '
          f'{outcome}')

runs_df = pd.DataFrame(runs)
summary_df = pd.DataFrame(summary)
runs_df.to_csv(RESULTS / "retrieval_runs.csv", index=False)
summary_df.drop(columns=["answer1", "answer2"]).to_csv(RESULTS / "query_summary.csv", index=False)
print(f"\nLLM calls used: {LLM_CALLS}/{LLM_CALL_BUDGET}  "
      f"(cache hits: {int(summary_df.a1_cache_hit.sum() + summary_df.a2_cache_hit.sum())})")
print(summary_df.outcome.value_counts().to_string())
print("\nanswers per model:")
print(summary_df.answer_model.value_counts().to_string())
missing = summary_df.loc[~summary_df.answers_generated, "qid"].tolist()
if missing:
    print(f"\nanswers still pending (retrieval evidence is complete): {', '.join(missing)}")

Q1 keyword_trap             gold rank    1 ->    1  changed_context_same_verdict


Q2 synonym_gap              gold rank    1 ->    1  changed_context_same_verdict


Q3 exception_corner         gold rank    4 ->    6  changed_context_same_verdict
Q4 negation_constraint      gold rank    6 ->    1  promotion


Q5 role_limit               gold rank    1 ->    1  changed_context_same_verdict
Q6 exception_corner         gold rank    3 ->    1  changed_context_same_verdict


Q7 negation_constraint      gold rank    1 ->    1  no_op_identical
Q8 exception_corner         gold rank    2 ->    2  changed_context_same_verdict


Q9 role_limit               gold rank    8 ->    1  promotion
Q10 negation_constraint      gold rank    7 ->    1  promotion


  candidate 'gemini-3.1-flash-lite' also unavailable: gemini-3.1-flash-lite: quota already known exhausted this run
    Q11: answers not generated (gemini-3.5-flash: quota already known exhausted this run)
Q11 keyword_trap             gold rank None -> None  changed_context_same_verdict
  candidate 'gemini-3.1-flash-lite' also unavailable: gemini-3.1-flash-lite: quota already known exhausted this run
    Q12: answers not generated (gemini-3.5-flash: quota already known exhausted this run)
Q12 exception_corner         gold rank    4 ->    2  promotion

LLM calls used: 0/26  (cache hits: 20)
outcome
changed_context_same_verdict    7
promotion                       4
no_op_identical                 1

answers per model:
answer_model
gemini-3.5-flash         8
gemini-3.1-flash-lite    2
unavailable              2

answers still pending (retrieval evidence is complete): Q11, Q12


## 6. Rank-shift analysis

Where the best gold chunk sat before versus after re-ranking. A blank baseline rank means it
never entered the Top 10, which re-ranking cannot fix — the cross-encoder only sees those ten.

In [13]:
shift = summary_df[["qid", "category", "gold_rules", "gold_rank_baseline", "gold_rank_rerank",
                    "outcome"]].copy()
# the brief's phrasing is "Chunk X moved from #8 -> #1", so carry the chunk id too
best_gold = {}
for q in QUERIES:
    d = runs_df[(runs_df.qid == q["qid"]) & runs_df.is_gold].sort_values("bi_rank")
    best_gold[q["qid"]] = int(d.chunk_id.iloc[0]) if len(d) else None
shift.insert(3, "best_gold_chunk", shift.qid.map(best_gold))
# pd.notna, not truthiness: a missing rank is NaN, and NaN is truthy
shift["shift"] = shift.apply(
    lambda r: (f'#{int(r.gold_rank_baseline)} -> #{int(r.gold_rank_rerank)}'
               if pd.notna(r.gold_rank_baseline) and pd.notna(r.gold_rank_rerank)
               else "gold never entered Top 10"), axis=1)
shift.to_csv(RESULTS / "rank_shifts.csv", index=False)
print(shift.to_string(index=False))

# "vital chunk buried at 4-10" is a rank property, not a question type: these are the queries
# that actually demonstrate it
buried = shift[(shift.gold_rank_baseline > TOP_K_CONTEXT) &
               (shift.gold_rank_rerank <= TOP_K_CONTEXT)]
print(f"\nburied 4-10 then promoted into the Top 3: {', '.join(buried.qid) or 'none'}")

# for each promotion, show what the cross-encoder pushed down to make room
for qid in shift.loc[shift.outcome == "promotion", "qid"]:
    d = runs_df[runs_df.qid == qid].sort_values("ce_rank")
    print(f"\n=== {qid}: {summary_df.set_index('qid').loc[qid, 'question']}")
    print(d[["bi_rank", "ce_rank", "rule_nos", "section_title", "page_from", "bi_score",
             "ce_score", "is_gold"]].to_string(index=False))

qid            category     gold_rules  best_gold_chunk  gold_rank_baseline  gold_rank_rerank                      outcome                     shift
 Q1        keyword_trap            2.4              6.0                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q2         synonym_gap      75.0;75.1            140.0                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q3    exception_corner 58.6;14.0;14.1            105.0                 4.0               6.0 changed_context_same_verdict                  #4 -> #6
 Q4 negation_constraint            5.2             11.0                 6.0               1.0                    promotion                  #6 -> #1
 Q5          role_limit      54.3;74.5             95.0                 1.0               1.0 changed_context_same_verdict                  #1 -> #1
 Q6    exception_corner      46.1;26.2             49.0                 3.0               1.0 changed_cont

## 7. Near-duplicate candidates from chunk overlap

Overlapping chunks can put the same rule in two candidates, spending two of the ten slots on
near-identical text. Logged because it is a direct cost of the overlap decision.

In [14]:
# pairs sharing rule numbers come from the overlap; pairs sharing none are the document
# stating the same rule twice
dups = []
for qid, grp in runs_df.groupby("qid"):
    ids = grp.chunk_id.tolist()
    in_top3 = dict(zip(grp.chunk_id, grp.in_baseline_top3))
    for i, a in enumerate(ids):
        for b in ids[i + 1:]:
            cos = float(corpus_emb[a] @ corpus_emb[b])
            if cos >= NEAR_DUP_COSINE:
                shared = set(str(chunks.loc[a, "rule_nos"]).split(";")) & \
                         set(str(chunks.loc[b, "rule_nos"]).split(";"))
                dups.append({"qid": qid, "chunk_a": a, "chunk_b": b, "cosine": round(cos, 4),
                             "shared_rules": ";".join(sorted(shared)) or "(none)",
                             "cause": "chunk overlap" if shared else "document repeats itself",
                             "both_in_baseline_top3": bool(in_top3[a] and in_top3[b])})
dups_df = pd.DataFrame(dups)
dups_df.to_csv(RESULTS / "near_duplicates.csv", index=False)
print(f"near-duplicate pairs in Top 10 (cosine >= {NEAR_DUP_COSINE}): {len(dups_df)}")
if len(dups_df):
    print(dups_df.to_string(index=False))

near-duplicate pairs in Top 10 (cosine >= 0.9): 9
qid  chunk_a  chunk_b  cosine shared_rules         cause  both_in_baseline_top3
 Q1      137      138  0.9318         73.1 chunk overlap                  False
Q10       83       84  0.9301         49.4 chunk overlap                  False
Q11      138      137  0.9318         73.1 chunk overlap                  False
Q12       76       75  0.9344         44.3 chunk overlap                  False
 Q3       44       45  0.9342         23.3 chunk overlap                  False
 Q5       55       54  0.9414         30.2 chunk overlap                  False
 Q7        9        8  0.9058          3.3 chunk overlap                   True
 Q9        9        8  0.9058          3.3 chunk overlap                   True
 Q9      137      138  0.9318         73.1 chunk overlap                  False


## 8. Answer 1 vs Answer 2

Written to `results/answers.md` side by side, with the model that produced each pair.

In [15]:
models_used = ", ".join(f"`{m}` ({n})" for m, n in summary_df.answer_model.value_counts().items())
lines = ["# Answer 1 (baseline) vs Answer 2 (re-ranked)", "",
         f"Models: {models_used} · temperature: provider default · "
         f"thinking: minimal · identical prompt template for both answers. Both answers of any "
         f"one query always come from the same model (asserted in §5).", ""]
for r in summary_df.itertuples():
    lines += [f"## {r.qid} — {r.category}", "", f"**Q:** {r.question}", "",
              f"*Model:* `{r.answer_model}` · *gold rules:* `{r.gold_rules}` · *gold rank:* "
              f"`#{r.gold_rank_baseline}` → `#{r.gold_rank_rerank}` · *outcome:* `{r.outcome}` · "
              f"*answers identical:* `{r.answers_identical}`", "",
              f"*Baseline Top 3 (chunk ids):* `{r.baseline_top3}` · "
              f"*Re-ranked Top 3:* `{r.rerank_top3}`", "",
              "**Answer 1**", "", r.answer1, "", "**Answer 2**", "", r.answer2, "", "---", ""]
(RESULTS / "answers.md").write_text("\n".join(lines))
print(f'wrote {RESULTS / "answers.md"} ({len(lines)} lines)')

for r in summary_df.itertuples():
    if not r.answers_identical:
        print(f"\n=== {r.qid} ({r.outcome}) {r.question}")
        print(f"--- Answer 1 (baseline top3 {r.baseline_top3}):\n{r.answer1}")
        print(f"--- Answer 2 (reranked top3 {r.rerank_top3}):\n{r.answer2}")

wrote /Users/d400l/Desktop/LMA_RAG/LMA_RAG/results/answers.md (220 lines)

=== Q1 (changed_context_same_verdict) How many cards is a player allowed to keep in their hand between rounds?
--- Answer 1 (baseline top3 6;137;91):
According to Rule 2.4, a player's hand can have a maximum of seven action cards, though a game effect can increase or decrease this limit. If a player ever has more than seven action cards, they must choose seven cards to keep and discard the rest (Rule 2.4).

**Rules cited:** Rule 2.4
--- Answer 2 (reranked top3 6;57;70):
According to Rule 2.4, a player’s hand can have a maximum of seven action cards, and if a player ever has more than seven, they must choose seven to keep and discard the rest. However, game effects can increase or decrease this hand limit. 

[Rule 2.4]

=== Q2 (changed_context_same_verdict) Can a warship fly into a star that has exploded?
--- Answer 1 (baseline top3 140;115;145):
No, a ship cannot move into a supernova (a star that has exploded).

## 9. Latency profiling

Each stage timed separately, `LATENCY_REPEATS` times per query, after a discarded warm-up.
Hardware goes in the table because these are wall-clock numbers from one laptop.

In [16]:
def time_ms(fn, n=LATENCY_REPEATS):
    fn()                                                  # warm-up, discarded
    out = []
    for _ in range(n):
        t0 = time.perf_counter(); fn(); out.append((time.perf_counter() - t0) * 1000)
    return out

timings = {"1. query embedding (bi-encoder)": [], "2. FAISS search (top 10 of N)": [],
           "3. cross-encoder scoring (10 pairs)": [],
           "Pipeline 1 retrieval total (1+2)": [],
           "Pipeline 2 retrieval total (1+2+3)": []}

for q in QUERIES:
    question = q["question"]
    qvec = bi_encoder.encode([question], convert_to_numpy=True,
                             normalize_embeddings=True).astype(np.float32)
    cands = baseline_retrieve(question)
    pairs = [(question, corpus_texts[c["chunk_id"]]) for c in cands]

    timings["1. query embedding (bi-encoder)"] += time_ms(
        lambda: bi_encoder.encode([question], convert_to_numpy=True, normalize_embeddings=True))
    timings["2. FAISS search (top 10 of N)"] += time_ms(
        lambda: index.search(qvec, TOP_K_RETRIEVE))
    timings["3. cross-encoder scoring (10 pairs)"] += time_ms(
        lambda: cross_encoder.predict(pairs, show_progress_bar=False))
    timings["Pipeline 1 retrieval total (1+2)"] += time_ms(
        lambda: baseline_retrieve(question))
    timings["Pipeline 2 retrieval total (1+2+3)"] += time_ms(
        lambda: rerank(question, baseline_retrieve(question)))

lat_rows = []
for stage, vals in timings.items():
    lat_rows.append({"stage": stage, "n": len(vals),
                     "mean_ms": round(statistics.mean(vals), 2),
                     "std_ms": round(statistics.stdev(vals), 2) if len(vals) > 1 else 0.0,
                     "median_ms": round(statistics.median(vals), 2),
                     "min_ms": round(min(vals), 2), "max_ms": round(max(vals), 2)})

# uncached calls only; averaging in cache hits would divide the real cost by the hit rate
# every cache entry stored the duration of the real request that produced it, so the cache is a
# better source than this run's uncached calls: it covers all generations instead of the few that
# happened to miss the cache today
llm_times = [v["seconds"] * 1000 for v in _cache.values() if v.get("seconds", 0) > 0]
if llm_times:
    lat_rows.append({"stage": "LLM generation (per real request, from cache)", "n": len(llm_times),
                     "mean_ms": round(statistics.mean(llm_times), 2),
                     "std_ms": round(statistics.stdev(llm_times), 2) if len(llm_times) > 1 else 0.0,
                     "median_ms": round(statistics.median(llm_times), 2),
                     "min_ms": round(min(llm_times), 2), "max_ms": round(max(llm_times), 2)})

latency = pd.DataFrame(lat_rows)
latency["hardware"] = HARDWARE
# the llm row is a weak statistic, so the csv carries that caveat itself
latency["note"] = ""
latency.loc[latency.stage.str.startswith("LLM"), "note"] = (
    f"durations of the {len(llm_times)} real generations, spanning "
    f"{summary_df.loc[summary_df.answers_generated, 'answer_model'].nunique()} model(s); "
    f"high variance, order of magnitude only")
latency.to_csv(RESULTS / "latency.csv", index=False)

print(f"hardware: {HARDWARE}")
print(f"corpus: {len(chunks)} chunks | queries: {len(QUERIES)} | repeats per query: "
      f"{LATENCY_REPEATS}\n")
print(latency.drop(columns=["hardware"]).to_string(index=False))

p1 = latency.loc[latency.stage.str.startswith("Pipeline 1"), "mean_ms"].iloc[0]
p2 = latency.loc[latency.stage.str.startswith("Pipeline 2"), "mean_ms"].iloc[0]
print(f"\nre-ranking adds {p2 - p1:.1f} ms ({p2 / p1:.2f}x) to retrieval for "
      f"{TOP_K_RETRIEVE} candidates")

hardware: arm64 / Darwin 25.4.0 / torch 2.5.1 / device=mps
corpus: 172 chunks | queries: 12 | repeats per query: 10

                                        stage   n  mean_ms  std_ms  median_ms  min_ms   max_ms                                                                                              note
              1. query embedding (bi-encoder) 120    15.75    0.54      15.67   14.73    18.10                                                                                                  
                2. FAISS search (top 10 of N) 120     0.01    0.00       0.01    0.01     0.03                                                                                                  
          3. cross-encoder scoring (10 pairs) 120   152.13   24.43     147.28  108.95   277.91                                                                                                  
             Pipeline 1 retrieval total (1+2) 120    16.56    1.96      15.87   14.79    25.72                 

## 10. Why not cross-encode the whole collection?

Measured, not asserted: cross-encoder throughput on this machine, extrapolated against what the
bi-encoder path costs per query.

In [17]:
# score the entire corpus against one query, which is the thing this section argues against
BULK = len(chunks)
bulk_pairs = [(QUERIES[0]["question"], t) for t in corpus_texts]
bulk_ms = time_ms(lambda: cross_encoder.predict(bulk_pairs, show_progress_bar=False), n=3)
pairs_per_sec = BULK / (statistics.mean(bulk_ms) / 1000)

embed_ms  = latency.loc[latency.stage.str.startswith("1."), "mean_ms"].iloc[0]
search_ms = latency.loc[latency.stage.str.startswith("2."), "mean_ms"].iloc[0]

scale_rows = []
for n_chunks in (len(chunks), 10_000, 1_000_000):
    ce_all = n_chunks / pairs_per_sec
    scale_rows.append({
        "corpus_chunks": n_chunks,
        "cross_encode_entire_corpus_s": round(ce_all, 2),
        "baseline_retrieval_s": round((embed_ms + search_ms) / 1000, 5),
        "rerank_top10_s": round(p2 / 1000, 5),
        "ce_all_vs_rerank_top10_x": round(ce_all / (p2 / 1000), 1)})
scaling = pd.DataFrame(scale_rows)
scaling["pairs_per_sec"] = round(pairs_per_sec, 1)
scaling["hardware"] = HARDWARE
scaling.to_csv(RESULTS / "scaling.csv", index=False)

print(f"cross-encoder throughput: {pairs_per_sec:.1f} (query, chunk) pairs/sec "
      f"[{BULK} pairs, mean of 3 runs, device={DEVICE}]\n")
print(scaling.drop(columns=["hardware"]).to_string(index=False))
print(f"\nThe bi-encoder embeds each chunk ONCE at index time; the cross-encoder cannot reuse "
      f"anything,\nbecause its score depends on the query and the chunk jointly. Its cost is "
      f"therefore O(N) per query,\nwhile FAISS search is sublinear and the two-stage pipeline "
      f"keeps the O(N) stage at N={TOP_K_RETRIEVE}.")

cross-encoder throughput: 67.5 (query, chunk) pairs/sec [172 pairs, mean of 3 runs, device=mps]

 corpus_chunks  cross_encode_entire_corpus_s  baseline_retrieval_s  rerank_top10_s  ce_all_vs_rerank_top10_x  pairs_per_sec
           172                          2.55               0.01576         0.16267                      15.7           67.5
         10000                        148.05               0.01576         0.16267                     910.1           67.5
       1000000                      14804.63               0.01576         0.16267                   91010.2           67.5

The bi-encoder embeds each chunk ONCE at index time; the cross-encoder cannot reuse anything,
because its score depends on the query and the chunk jointly. Its cost is therefore O(N) per query,
while FAISS search is sublinear and the two-stage pipeline keeps the O(N) stage at N=10.


## 11. Outputs

Everything above is written to `results/` — `chunks.csv`, `retrieval_runs.csv`,
`query_summary.csv`, `rank_shifts.csv`, `failure_hunt.csv`, `near_duplicates.csv`,
`answers.md`, `answers_cache.json`, `latency.csv`, `scaling.csv`.

`docs/CELL_GUIDE.md` carries the cell-by-cell detail. The report is written by hand from these.